# Lab: Streaming Agent Responses

This notebook demonstrates **how to stream outputs from a LangChain agent** — letting you
see tokens and events in real-time rather than waiting for the full response.

Streaming is crucial for good UX in LLM-powered apps: instead of a long pause followed by
a wall of text, users see the response being written token-by-token.

## Structure

| Section | What you will learn |
|---------|---------------------|
| **Shared Setup** | Build a simple agent with mock tools using `create_agent` (no extra API keys needed) |
| **Version 1 — Token Streaming** | Stream the LLM output token by token using `stream_mode="messages"` |
| **Version 2 — Tool Call Streaming** | *(Your turn!)* Stream agent progress and tool calls using `stream_mode="updates"` |

> **Reference:** [LangChain Streaming docs](https://docs.langchain.com/oss/python/langchain/streaming#agent-progress)

---

### Key concept: stream modes

The agent's `.stream()` method accepts a `stream_mode` parameter:

| Mode | What it streams |
|------|----------------|
| `"messages"` | LLM tokens one by one, with metadata |
| `"updates"` | State changes after each node finishes (agent progress) |
| `"custom"` | Arbitrary data emitted from nodes via `get_stream_writer` |

We will use `version="v2"` throughout — it gives a **unified `StreamPart` format**
so every chunk looks the same regardless of mode:

```python
{
    "type": "messages" | "updates" | "custom" | ...,
    "ns":   (),        # namespace (populated for subgraphs)
    "data": ...,       # the actual payload
}
```

### Node names in `create_agent`

When using `langchain.agents.create_agent`, the graph has two nodes:

| Node name | What it does |
|-----------|-------------|
| `"model"` | The LLM — decides which tools to call, then synthesises the final answer |
| `"tools"` | Executes the tool calls and returns results |

## Shared Setup (Run Once)

1. Set `MODEL_PROVIDER = "openai"` or `"gemini"` below.
2. Run this cell before anything else.
3. The agent is built with two **mock tools** — no Tavily key needed.
   - `get_weather(city)` returns a fake weather report
   - `calculate(expression)` evaluates a simple math expression

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("LLM ready — provider: %s", MODEL_PROVIDER)

# ── Mock tools (no external API needed) ───────────────────────────────────────
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    logger.debug("get_weather called with city=%s", city)
    fake_data = {
        "hanoi":     "Hanoi: 32°C, humid, partly cloudy.",
        "ho chi minh city": "Ho Chi Minh City: 34°C, sunny.",
        "london":    "London: 15°C, overcast with light rain.",
        "new york":  "New York: 22°C, clear skies.",
        "tokyo":     "Tokyo: 18°C, cherry blossoms in bloom.",
    }
    return fake_data.get(city.lower(), f"{city}: 25°C, mostly sunny.")


@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the result."""
    logger.debug("calculate called with expression=%s", expression)
    try:
        # Only allow safe arithmetic — no builtins, no imports
        result = eval(expression, {"__builtins__": {}})  # noqa: S307
        return f"{expression} = {result}"
    except Exception as exc:
        return f"Error evaluating '{expression}': {exc}"


tools = [get_weather, calculate]

# ── Build the ReAct agent ─────────────────────────────────────────────────────
# create_react_agent wires: START → agent_node ↔ tool_node → END
agent = create_agent(llm, tools)

logger.info("Agent ready with tools: %s", [t.name for t in tools])

INFO | LLM ready — provider: openai
INFO | Agent ready with tools: ['get_weather', 'calculate']


---
## Version 1 — Stream LLM Tokens (Token-by-Token)

### How it works

Use `stream_mode="messages"` to receive **(token_chunk, metadata)** pairs as the LLM
generates its response. Each `chunk` in the stream looks like:

```python
{
    "type": "messages",
    "ns":   (),
    "data": (message_chunk, metadata_dict)
}
```

- **`message_chunk.content`** — the token text (may be an empty string for non-text chunks, always check)
- **`metadata["langgraph_node"]`** — the node that produced this token (`"model"` or `"tools"`)

### Filter to only LLM tokens

With `create_agent`, the `"messages"` stream includes tokens from both nodes.
Filter on `metadata["langgraph_node"] == "model"` to see only what the LLM outputs:

```python
node: model   ← tool-call decision tokens (JSON args being built up)
node: tools   ← tool result message
node: model   ← final answer tokens
```

> **Try it:** change the question to `"What is 42 * 37?"` to see the calculator tool fire.

In [2]:
# ── Version 1: Stream LLM tokens ─────────────────────────────────────────────

question = "What is the weather in Hanoi and Tokyo? Also calculate 15 * 8 for me."

logger.info("Starting Version 1 stream for question: %s", question)
print(f"Question: {question}")
print("=" * 60)
print("Streaming all LLM tokens (tool-call args + final answer):")
print()

for chunk in agent.stream(
    {"messages": [HumanMessage(content=question)]},
    stream_mode="messages",
    version="v2",
):
    # Every chunk is a StreamPart dict — check the type first
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]

        # Filter: only show tokens from the 'model' node (the LLM)
        # The 'tools' node emits completed ToolMessage results, not streaming tokens
        if metadata["langgraph_node"] == "model" and message_chunk.content:
            print(message_chunk.content, end="", flush=True)

print()  # newline after the streamed response
print()
logger.info("Version 1 stream complete")

INFO | Starting Version 1 stream for question: What is the weather in Hanoi and Tokyo? Also calculate 15 * 8 for me.


Question: What is the weather in Hanoi and Tokyo? Also calculate 15 * 8 for me.
Streaming all LLM tokens (tool-call args + final answer):



INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The current weather is as follows:
- **Hanoi**: 32°C, humid, partly cloudy.
- **Tokyo**: 18°C, cherry blossoms in bloom.

Additionally, the result of the calculation \( 15 \times 8 \) is **120**.

INFO | Version 1 stream complete


### What just happened?

The graph executed roughly like this:

```
START
  └─► model node  (LLM decides to call get_weather + calculate → streams tool-call JSON)
        └─► tools node  (executes both tools, returns results)
              └─► model node  (LLM reads results → streams final answer tokens)
                    └─► END
```

The `"messages"` stream mode emits a chunk for **every token** the LLM outputs — both
the tool-call argument JSON and the final answer. We filtered by
`metadata["langgraph_node"] == "model"` to see only LLM-generated tokens.

> **Key insight:** filtering on `langgraph_node` lets you selectively stream tokens
> from specific nodes in your graph.

---
## Version 2 — Stream Agent Progress & Tool Calls *(Your Turn!)*

In Version 1 we streamed LLM tokens. Now you will use `stream_mode="updates"` to track
**agent progress step by step** — which node ran and what it produced.

Per the [LangChain streaming docs](https://docs.langchain.com/oss/python/langchain/streaming#agent-progress),
if the agent calls one tool you should see three update events:

| Step | Node | Content |
|------|------|---------|
| 1 | `model` | `AIMessage` with `.tool_calls` populated |
| 2 | `tools` | `ToolMessage` with the tool result |
| 3 | `model` | Final `AIMessage` with the answer |

Your goal: combine `"updates"` and `"messages"` to print **which tools are called** and
then **stream the final answer token-by-token**, so the output looks like:

```
Question: What is the weather in Hanoi and Tokyo? Also calculate 15 * 8 for me.
============================================================
[Step] model
  → Tool call: get_weather  args: {'city': 'Hanoi'}
  → Tool call: get_weather  args: {'city': 'Tokyo'}
  → Tool call: calculate    args: {'expression': '15 * 8'}
[Step] tools
  → Tool result: Hanoi: 32°C, humid, partly cloudy.
  → Tool result: Tokyo: 18°C, cherry blossoms in bloom.
  → Tool result: 15 * 8 = 120
[Step] model — streaming final answer:
The weather in Hanoi is... <token by token>
```

### Hints

1. **Use `stream_mode=["messages", "updates"]`** to receive both types of chunks.

2. **`"updates"` chunks** have this shape:
   ```python
   # chunk["type"] == "updates"
   # chunk["data"] == {"node_name": {"messages": [...]}}
   for source, update in chunk["data"].items():
       last_msg = update["messages"][-1]
       print(f"[Step] {source}")
   ```

3. **Check the message type** to know what happened at each step:
   ```python
   from langchain_core.messages import AIMessage, ToolMessage

   if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
       for tc in last_msg.tool_calls:
           print(f"  → Tool call: {tc['name']}  args: {tc['args']}")

   if isinstance(last_msg, ToolMessage):
       print(f"  → Tool result: {last_msg.content}")
   ```

4. **For `"messages"` chunks**, stream the final answer exactly as in Version 1:
   filter on `metadata["langgraph_node"] == "model"` and print non-empty content.

### TODO: Complete the code below

In [ ]:
# ── Version 2: Stream agent progress + LLM tokens ────────────────────────────
from langchain_core.messages import AIMessage, ToolMessage

question = "What is the weather in Hanoi and Tokyo? Also calculate 15 * 8 for me."

logger.info("Starting Version 2 stream for question: %s", question)
print(f"Question: {question}")
print("=" * 60)

for chunk in agent.stream(
    {"messages": [HumanMessage(content=question)]},
    # TODO: pass a list with both "messages" and "updates" as stream_mode
    stream_mode=["messages", "updates"],  # ← replace None with ["messages", "updates"]
    version="v2",
):
    # TODO: handle "updates" chunks to print agent progress
    # Hint: iterate over chunk["data"].items() to get (source, update) pairs
    # - source is the node name: "model" or "tools"
    # - update["messages"][-1] is the last message produced by that node
    # - Check isinstance(msg, AIMessage) and msg.tool_calls to find tool calls
    # - Check isinstance(msg, ToolMessage) to find tool results
    if chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            last_msg = update["messages"][-1]
            
            if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
                print(f"[Step] {source}")
                for tc in last_msg.tool_calls:
                    print(f"  → Tool call: {tc['name']}  args: {tc['args']}")
            
            elif isinstance(last_msg, ToolMessage):
                print(f"[Step] {source}")
                print(f"  → Tool result: {last_msg.content}")
                
            elif isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
                # This is the final answer from the model node
                print(f"[Step] {source} — streaming final answer:")

    # TODO: handle "messages" chunks to stream the LLM tokens
    # Hint: same as Version 1 — filter on metadata["langgraph_node"] == "model"
    # and only print when message_chunk.content is non-empty
    elif chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        
        # Filter: only show tokens from the 'model' node (the LLM)
        # The 'tools' node emits completed ToolMessage results, not streaming tokens
        if metadata["langgraph_node"] == "model" and message_chunk.content:
            print(message_chunk.content, end="", flush=True)

print()
logger.info("Version 2 stream complete")

### Expected output

```
Question: What is the weather in Hanoi and Tokyo? Also calculate 15 * 8 for me.
============================================================
[Step] model
  → Tool call: get_weather  args: {'city': 'Hanoi'}
  → Tool call: get_weather  args: {'city': 'Tokyo'}
  → Tool call: calculate    args: {'expression': '15 * 8'}
[Step] tools
  → Tool result: Hanoi: 32°C, humid, partly cloudy.
  → Tool result: Tokyo: 18°C, cherry blossoms in bloom.
  → Tool result: 15 * 8 = 120
[Step] model — streaming final answer:
The weather in Hanoi is 32°C... <continues token by token>
```

---
## Bonus Challenges

1. **Timestamp each tool call** — measure how long the `tools` step takes.
   (Hint: record `time.time()` when you see the `model` update with tool calls,
   then again when you see the `tools` update, and print the difference.)

2. **Stream custom data from inside a tool** — add `get_stream_writer()` inside
   `get_weather` to emit a progress message before returning. Use
   `stream_mode=["updates", "custom"]` to receive it.
   See: [Custom updates docs](https://docs.langchain.com/oss/python/langchain/streaming#custom-updates)

3. **Stream only the final answer** — modify Version 1 so it shows *nothing* during
   the tool-call phase and only starts printing when the second `"model"` step begins.
   (Hint: track a `tool_calls_seen` boolean flag.)